In [1]:
# !pip -q install polars 

In [2]:
# !pip -q install fastexcel


**Step 1: Data Ingestion and Schema Standardization**
Disparate datasets containing demographic profiles, socioeconomic indicators, and components of population change are merged into a singular, unified framework. All incoming records are structurally aligned to ensure consistent representation of the reporting year, the specific metric, the geographic entity, the numerical value, and the comparison category (such as a peer group or state benchmark).

**Step 2: Value Formatting and Anomaly Detection**
Raw numerical figures undergo a transformation into human-readable strings tailored to the metric type. Appropriate symbols—such as percentages, currency indicators, millions designations, or directional signs (plus/minus) for year-over-year changes—are applied. A logic mechanism actively monitors for anomalies, such as massive absolute counts incorrectly flagged as percentages, automatically reverting them to standard numeric formats to maintain data integrity.

**Step 3: Metric Calculation and Temporal Alignment**
The standardized data is routed through specialized mathematical engines depending on the nature of the metric:
* **Drivers of Change:** Migration and natural birth/death components are aggregated to identify the leading contributor to population shifts.
* **Cumulative Tracking:** Current values are subtracted from baseline historical years (e.g., 1990) to determine long-term growth.
* **Demographic Grouping:** Age brackets and race/ethnicity cohorts are sorted to pinpoint the largest segments.
* **Growth Rates:** Year-over-year deltas and percentage changes are computed between the most recent available periods. 
Concurrently, the exact baseline and current years are extracted to establish precise temporal context.

**Step 4: Mathematical Constraint Generation**
To guarantee analytical accuracy, precise mathematical relationships between a primary geographic area and its comparative benchmarks are calculated programmatically. These unalterable facts—whether they are percentage point differences, exact numeric margins, or direct ratios—are generated as hard constraints to govern the text-generation phase, preventing any hallucination of mathematical figures.

**Step 5: Contextual Prompt Assembly**
Highly structured instructional templates are constructed for a language model. These templates combine the formatted geographic data, the mathematical constraints, and stringent stylistic directives. Tailored examples are injected into the template based on the metric category (percentage, numerical, categorical, or ordinal). This dynamic injection dictates appropriate vocabulary, actively restricting competitive verbs (like "outpacing") when comparing fundamentally different categories, such as distinct age brackets.

**Step 6: Insight Generation and Precision Extraction**
The assembled instructions are processed by the language model, which is restricted to outputting structured JSON format. To counter any conversational tendencies of the model, a targeted regular expression mechanism scans the output, surgically extracting only the requested data block and systematically eliminating any introductory phrases, conversational filler, or trailing syntax errors.

**Step 7: Sequential Quality Assurance Layers**
The extracted insights are passed through two consecutive, automated editorial filters to ensure production-level quality:
* **Grammatical Consistency:** Enforces strict lowercase formatting for demographic categories, age ranges, and economic indicators, ensuring visual uniformity.
* **Geographic Standardization:** Cross-references every geographic name against an approved master list. This layer corrects unauthorized abbreviations, ensures proper Title Case for locations, and prevents distinct regional names from being erroneously merged or substituted.

**Step 8: Directional Insight Synthesis**
Four distinct analytical perspectives—internal trends, broad regional comparisons, state/national benchmarks, and peer averages—are aggregated for a final synthesis. A comprehensive "reference matrix" containing all raw formatted values is provided alongside the text. This matrix acts as a cheat sheet, allowing the deduction of purely directional relationships (e.g., simply identifying if a value is higher or lower) to weave the four perspectives into one cohesive summary sentence without fabricating cross-comparative math.

**Step 9: Final Compilation and Export**
All generated insights, alongside their corresponding metadata, periods of comparison, and metric definitions, are compiled into a final structured format. The data is aligned into precise columns and ultimately exported as a flat file, ready for seamless integration into a professional reporting dashboard.

In [3]:
# import polars as pl
# import pandas as pd
# import ollama
# import time
# import json
# import random 
# import re 

# MODEL_NAME = "gemma3" 

# # ==========================================
# # 1. STANDARDIZATION & FORMATTING
# # ==========================================
# def standardize_dataset(df: pl.DataFrame, source_type: str) -> pl.DataFrame:
#     if source_type == "ACS":
#         return df.select([pl.col("Year").cast(pl.Int64), pl.col("Cleaned Field Name").alias("Metric"), pl.col("NAME"), pl.col("Value").cast(pl.Float64), pl.col("Role")])
#     elif source_type == "COMPONENTS":
#         return df.select([pl.col("Year").cast(pl.Int64), pl.col("Value Type").alias("Metric"), pl.col("NAME"), pl.col("Values").alias("Value").cast(pl.Float64), pl.col("Role")])
#     elif source_type == "POP_PYRAMID":
#         return df.select([pl.col("YEAR").cast(pl.Int64).alias("Year"), (pl.col("Variable Group Description") + " (" + pl.col("Age Group Description") + ")").alias("Metric"), pl.col("NAME"), pl.col("Values").alias("Value").cast(pl.Float64), pl.col("Role")])

# def format_value(value, metric_name, m_type=""):
#     try:
#         val = float(value)
#         if val != val: return "N/A"
#     except: return str(value)
    
#     m_lower = str(metric_name).lower()
#     type_lower = str(m_type).lower()
    
#     is_percentage = False
#     if "percent" in type_lower or "rate" in type_lower:
#         is_percentage = True
#     elif ("percent" in m_lower or "rate" in m_lower) and "numeric" not in type_lower:
#         is_percentage = True
        
#     if is_percentage and abs(val) > 100:
#         is_percentage = False
    
#     if is_percentage:
#         if abs(val) <= 1.0 and val != 0:
#             return f"{val * 100:.1f}%"
#         else:
#             return f"{val:.1f}%"
            
#     elif "age" in m_lower and "largest" not in m_lower:  
#         if val == 0: return "0.0"
#         return f"+{val:.1f}" if ("change" in m_lower and val > 0) else f"{val:.1f}"
#     elif "income" in m_lower or "cost" in m_lower: 
#         return f"${val:,.0f}"
#     elif "change" in m_lower or "net" in m_lower or "mig" in m_lower or "driver" in m_lower: 
#         if val == 0: return "0"
#         return f"+{val:,.0f}" if val > 0 else f"{val:,.0f}"
#     else: 
#         if val >= 1_000_000: return f"{val/1_000_000:.2f} million"
#         return f"{val:,.0f}"

# # ==========================================
# # 2. REQUIRED FACT GENERATOR (Math Lock)
# # ==========================================
# def get_required_fact(fv_raw, cv_raw, cn, metric_name, m_type, is_categorical=False, f_cat=None, c_cat=None):
#     try:
#         f = float(fv_raw)
#         c = float(cv_raw)
#         if c == 0 and f != 0: return ""
        
#         m_lower = str(metric_name).lower()
#         type_lower = str(m_type).lower()
#         cn_first = cn.replace("Peers (Average)", "its peer average").replace("Peers (Combined)", "its combined peers").split(" and ")[0] 
        
#         if is_categorical:
#             if f_cat == c_cat:
#                 diff = f - c
#                 return f"REQUIRED FACT: You must state both share the primary group '{f_cat}', but the magnitude differs by {abs(diff):,.0f}."
#             else:
#                 return f"REQUIRED FACT: You must explicitly state that the primary group is '{f_cat}', whereas for {cn_first} it is '{c_cat}'. DO NOT subtract their values."
        
#         if f == c:
#             return f"REQUIRED FACT: You must explicitly state that the values are exactly identical. DO NOT say it is 0 points higher or lower."

#         is_percentage = False
#         if "percent" in type_lower or "rate" in type_lower:
#             is_percentage = True
#         elif ("percent" in m_lower or "rate" in m_lower) and "numeric" not in type_lower:
#             is_percentage = True
            
#         if is_percentage and (abs(f) > 100 or abs(c) > 100):
#             is_percentage = False
            
#         if m_lower in ["median age", "change in median age"]:
#             diff = f - c
#             return f"REQUIRED FACT: You must state the difference is exactly {abs(diff):.1f} years {'older' if diff > 0 else 'younger'} than {cn_first}."
#         elif is_percentage:
#             diff = f - c
#             return f"REQUIRED FACT: You must state the difference is exactly {abs(diff)*100:.1f} percentage points {'higher' if diff > 0 else 'lower'} than {cn_first}. DO NOT use the '%' symbol for the difference."
#         elif "change" in m_lower or "net" in m_lower or "mig" in m_lower or "driver" in m_lower:
#             diff = f - c
#             return f"REQUIRED FACT: You must state the difference is exactly {abs(diff):,.0f} {'more' if diff > 0 else 'fewer'} than {cn_first}."
#         else:
#             ratio = f / c
#             if ratio < 1: return f"REQUIRED FACT: You must state it is approximately {ratio*100:.1f}% the size of {cn_first}."
#             else: return f"REQUIRED FACT: You must state it is approximately {ratio:.1f} times larger than {cn_first}."
#     except:
#         return ""

# # ==========================================
# # 3. UNIFIED DATA ENGINE
# # ==========================================
# def combine_roles(df_result, value_col, metric_name, m_type, is_categorical=False, category_col=None):
#     geo_data = {}
#     if not is_categorical:
#         df_result = df_result.sort(value_col) 
        
#     role_dict = {}
#     for r in df_result.iter_rows(named=True):
#         role = r["Role"]
#         if role != "Peer":
#             if role not in role_dict: role_dict[role] = []
#             role_dict[role].append(r)
            
#     for role, rows in role_dict.items():
#         if len(rows) == 1:
#             r = rows[0]
#             if is_categorical: 
#                 geo_data[role] = {"Name": r["NAME"], "Formatted_Value": f"the '{r[category_col]}' group with {format_value(r[value_col], metric_name, m_type)}", "Raw_Value": r[value_col], "Category": r[category_col]}
#             else: 
#                 geo_data[role] = {"Name": r["NAME"], "Formatted_Value": format_value(r[value_col], metric_name, m_type), "Raw_Value": r[value_col]}
#         else:
#             combined_name = " and ".join([r["NAME"] for r in rows])
#             if is_categorical: 
#                 combined_formatted = " and ".join([f"{r['NAME']}: the '{r[category_col]}' group with {format_value(r[value_col], metric_name, m_type)}" for r in rows])
#                 geo_data[role] = {"Name": combined_name, "Formatted_Value": combined_formatted, "Raw_Value": rows[0][value_col], "Category": rows[0][category_col]}
#             else: 
#                 combined_formatted = " and ".join([f"{r['NAME']}: {format_value(r[value_col], metric_name, m_type)}" for r in rows])
#                 geo_data[role] = {"Name": combined_name, "Formatted_Value": combined_formatted, "Raw_Value": rows[0][value_col]}
            
#     df_peer = df_result.filter(pl.col("Role") == "Peer")
#     if not df_peer.is_empty():
#         if is_categorical:
#             r = df_peer.row(0, named=True)
#             geo_data["Peer"] = {"Name": r["NAME"], "Formatted_Value": f"the '{r[category_col]}' group with {format_value(r[value_col], metric_name, m_type)}", "Raw_Value": r[value_col], "Category": r[category_col]}
#         else:
#             peer_avg = df_peer[value_col].mean()
#             geo_data["Peer"] = {"Name": "Peers (Average)", "Formatted_Value": format_value(peer_avg, metric_name, m_type), "Raw_Value": peer_avg}
            
#     return geo_data

# def calculate_metric_data(master_df, df_pyr, metric_name, variables_json, m_type):
#     m_lower = metric_name.lower()
#     type_lower = str(m_type).lower()
    
#     is_yoy_change = ("change" in m_lower and "cumulative" not in m_lower)
#     is_percentage_metric = "percent" in type_lower or "rate" in type_lower or (("percent" in m_lower or "rate" in m_lower) and "numeric" not in type_lower)
    
#     years_context = {"latest": "", "prev": ""}
    
#     if "driver" in m_lower or "dynamics in drivers" in m_lower:
#         is_cumulative = "cumulative" in m_lower
#         drivers = ["NATURALCHG", "DOMESTICMIG", "INTERNATIONALMIG"]
#         df_d = master_df.filter(pl.col("Metric").is_in(drivers))
        
#         if df_d.is_empty(): return None, years_context, True
        
#         years = df_d["Year"].drop_nulls().unique().sort()
#         if len(years) > 0: years_context["latest"] = str(years[-1])
#         if len(years) > 1: years_context["prev"] = str(years[-2])
        
#         if is_cumulative: df_d = df_d.filter(pl.col("Year") >= 1990)
#         else: df_d = df_d.filter(pl.col("Year") == df_d["Year"].max())
#         if df_d.is_empty(): return None, years_context, True
        
#         df_std = df_d.filter(pl.col("Role") != "Peer")
#         df_peer = df_d.filter(pl.col("Role") == "Peer").with_columns(pl.lit("Peers (Combined)").alias("NAME"))
#         df_peer = df_peer.group_by(["NAME", "Role", "Metric"]).agg(pl.col("Value").sum())
#         df_combined = pl.concat([df_std.select(["NAME", "Role", "Metric", "Value"]), df_peer])
        
#         agg = df_combined.group_by(["NAME", "Role", "Metric"]).agg(pl.col("Value").sum())
#         largest = agg.with_columns(pl.col("Value").abs().alias("Abs_Val")).sort("Abs_Val", descending=True).group_by(["NAME", "Role"]).first()
        
#         names = {"NATURALCHG": "Natural Change", "DOMESTICMIG": "Domestic Migration", "INTERNATIONALMIG": "International Migration"}
#         largest = largest.with_columns(pl.col("Metric").replace(names).alias("Driver_Name"))
        
#         return combine_roles(largest, "Value", metric_name, m_type, is_categorical=True, category_col="Driver_Name"), years_context, True

#     elif "cumulative" in m_lower and "population" in m_lower and "change" in m_lower:
#         df_pop = master_df.filter(pl.col("Metric") == "POPESTIMATE")
#         if df_pop.is_empty() or df_pop.filter(pl.col("Year") == 1990).is_empty(): return None, years_context, False
        
#         years = df_pop["Year"].drop_nulls().unique().sort()
#         if len(years) > 0: years_context["latest"] = str(years[-1])
#         if len(years) > 1: years_context["prev"] = str(years[-2])
        
#         latest_year = df_pop["Year"].max()
#         df_latest = df_pop.filter(pl.col("Year") == latest_year).select(["NAME", "Role", "Value"])
#         df_1990 = df_pop.filter(pl.col("Year") == 1990).select(["NAME", "Role", "Value"])
        
#         change_df = df_latest.join(df_1990, on=["NAME", "Role"], how="inner")
#         if is_percentage_metric:
#             change_df = change_df.with_columns(((pl.col("Value") - pl.col("Value_right")) / pl.col("Value_right")).alias("Change"))
#         else:
#             change_df = change_df.with_columns((pl.col("Value") - pl.col("Value_right")).alias("Change"))
            
#         return combine_roles(change_df, "Change", metric_name, m_type), years_context, False

#     elif "largest" in m_lower and ("age" in m_lower or "race" in m_lower):
#         years = df_pyr["YEAR"].drop_nulls().unique().sort()
#         if len(years) == 0: return None, years_context, True
        
#         latest = years[-1]
#         years_context["latest"] = str(latest)
#         if len(years) > 1: years_context["prev"] = str(years[-2])
        
#         if "race" in m_lower:
#             df_pyr_clean = df_pyr.with_columns(pl.col("Variable Group Description").str.replace(" Male", "").str.replace(" Female", "").alias("Group_Clean"))
#             df_pyr_clean = df_pyr_clean.filter(~pl.col("Group_Clean").is_in(["Male", "Female", "Total", "All"]))
#             group_col = "Group_Clean"
#         else:
#             df_pyr_clean = df_pyr.filter(pl.col("Age Group Description") != "All")
#             group_col = "Age Group Description"
            
#         df_std = df_pyr_clean.filter(pl.col("Role") != "Peer")
#         df_peer = df_pyr_clean.filter(pl.col("Role") == "Peer").with_columns(pl.lit("Peers (Combined)").alias("NAME"))
#         df_peer = df_peer.group_by(["NAME", "Role", "YEAR", group_col]).agg(pl.col("Values").sum())
#         df_combined = pl.concat([df_std.select(["NAME", "Role", "YEAR", group_col, "Values"]), df_peer])

#         if not is_yoy_change:
#             df_l = df_combined.filter(pl.col("YEAR") == latest).group_by(["NAME", "Role", group_col]).agg(pl.col("Values").cast(pl.Float64).sum())
#             largest = df_l.sort("Values", descending=True).group_by(["NAME", "Role"]).first()
#             return combine_roles(largest, "Values", metric_name, m_type, is_categorical=True, category_col=group_col), years_context, True
#         else:
#             if len(years) < 2: return None, years_context, True
#             prev = years[-2] 
            
#             df_l = df_combined.filter(pl.col("YEAR") == latest).group_by(["NAME", "Role", group_col]).agg(pl.col("Values").cast(pl.Float64).sum().alias("VL"))
#             df_p = df_combined.filter(pl.col("YEAR") == prev).group_by(["NAME", "Role", group_col]).agg(pl.col("Values").cast(pl.Float64).sum().alias("VP"))
#             change = df_l.join(df_p, on=["NAME", "Role", group_col], how="inner").with_columns((pl.col("VL") - pl.col("VP")).alias("Chg"))
#             largest = change.sort("Chg", descending=True).group_by(["NAME", "Role"]).first()
#             return combine_roles(largest, "Chg", metric_name, m_type, is_categorical=True, category_col=group_col), years_context, True

#     else:
#         target_raw = metric_name 
#         if variables_json and isinstance(variables_json, str):
#             try: target_raw = list(json.loads(variables_json.replace('""', '"')).values())[0] 
#             except: pass
                
#         df_metric = master_df.filter(pl.col("Metric") == target_raw)
#         if df_metric.is_empty(): return None, years_context, False
            
#         years = df_metric["Year"].drop_nulls().unique().sort()
#         if len(years) > 0: years_context["latest"] = str(years[-1])
#         if len(years) > 1: years_context["prev"] = str(years[-2])
            
#         latest_year = df_metric["Year"].max()
        
#         if is_yoy_change:
#             if len(years) < 2: return None, years_context, False
#             prev_year = years[-2]
            
#             df_l_dedup = df_metric.filter(pl.col("Year") == latest_year).group_by(["NAME", "Role"]).agg(pl.col("Value").mean())
#             df_p_dedup = df_metric.filter(pl.col("Year") == prev_year).group_by(["NAME", "Role"]).agg(pl.col("Value").mean())
#             change_df = df_l_dedup.join(df_p_dedup, on=["NAME", "Role"], how="inner")
            
#             max_val = change_df["Value_right"].max()
#             if is_percentage_metric and max_val and max_val > 100:
#                 change_df = change_df.with_columns(((pl.col("Value") - pl.col("Value_right")) / pl.col("Value_right")).alias("Change"))
#             else:
#                 change_df = change_df.with_columns((pl.col("Value") - pl.col("Value_right")).alias("Change"))
                
#             return combine_roles(change_df, "Change", metric_name, m_type), years_context, False
#         else:
#             df_dedup = df_metric.filter(pl.col("Year") == latest_year).group_by(["NAME", "Role"]).agg(pl.col("Value").mean())
#             return combine_roles(df_dedup, "Value", metric_name, m_type), years_context, False


# # ==========================================
# # 4. LLM INTERFACE & PROMPTING
# # ==========================================
# def get_ollama_text(prompt, as_json=False, json_key="overall_insight"):
#     if not prompt: return "N/A"
#     try:
#         resp = ollama.generate(
#             model=MODEL_NAME, 
#             prompt=prompt, 
#             options={
#                 "temperature": 0.1, 
#                 "seed": random.randint(1, 100000),
#                 "num_predict": 150
#             }
#         )['response'].strip()
        
#         if as_json:
#             match = re.search(r'\{.*?\}', resp, re.DOTALL)
#             if match:
#                 try:
#                     val = json.loads(match.group(0)).get(json_key, "JSON Error")
#                 except:
#                     val = f"JSON Parse Error: {match.group(0)}"
#             else:
#                 val = f"Regex Error: No JSON found"
#         else:
#             val = resp

#         val = re.sub(r"^.*?(summary|sentence|data|revised|professional).*?:", "", val, flags=re.IGNORECASE).strip()
#         val = val.rstrip('"}” \n\r\t')
        
#         return val.replace('\n', ' ')
#     except Exception as e: 
#         return f"Error: {e}"

# # ==========================================
# # 5. POST-PROCESSING (Grammar & Geography)
# # ==========================================
# def apply_grammar_layer(text):
#     if text in ["N/A", "JSON Error", "JSON Parse Error", "JSON Format Error", ""] or str(text).startswith("Error:") or str(text).startswith("Regex Error:"): return text
    
#     p = f"""You are a strict copyeditor. Fix capitalization and grammar in the sentence below.
# CRITICAL RULES:
# 1. Demographic categories, age groups, and drivers of change MUST be lowercase (e.g., 'international migration', 'domestic migration', 'natural change', 'white', 'hispanic') unless they start a sentence.
# 2. Geographic locations MUST be fully capitalized (e.g., 'Texas', 'United States', 'Austin MSA'). Do NOT lowercase them.
# 3. Output STRICTLY as a valid JSON object. Do not include intro text.

# Original Sentence: {text}
# Output format: {{ "revised_sentence": "your fixed sentence here" }}
# """
#     return get_ollama_text(p, as_json=True, json_key="revised_sentence")

# def apply_geography_standardization_layer(text, valid_geos):
#     if text in ["N/A", "JSON Error", "JSON Parse Error", "JSON Format Error", ""] or str(text).startswith("Error:") or str(text).startswith("Regex Error:"): return text
    
#     valid_geos_clean = list(set([g.replace("Peers (Average)", "its peer average").replace("Peers (Combined)", "its combined peers") for g in valid_geos if g]))
#     geo_str = ", ".join([f"'{g}'" for g in valid_geos_clean])
    
#     p = f"""You are a strict copyeditor. Ensure the geography names in the sentence below EXACTLY match the provided allowed list.
# CRITICAL RULES:
# 1. The valid geography names for this sentence are EXACTLY: {geo_str}.
# 2. Replace any ALL-CAPS names (like 'TEXAS' or 'UNITED STATES') with their proper Title Case format as shown in the valid list.
# 3. Replace any unauthorized abbreviations (like 'Austin MSA') with the full name from the valid list, if applicable.
# 4. VERY IMPORTANT: Do NOT change one distinct geography into another (e.g., do not replace "Texas" with "Travis County, Texas"). Treat overlapping names as distinct entities.
# 5. Output STRICTLY as a valid JSON object. Do not include intro text.

# Original Sentence: {text}
# Output format: {{ "revised_sentence": "your fixed sentence here" }}
# """
#     return get_ollama_text(p, as_json=True, json_key="revised_sentence")

# # ==========================================
# # 6. PIPELINE EXECUTION
# # ==========================================
# def create_prompt(insight_type, geo_data, metric_name, metric_desc, years_ctx, is_categorical, m_type):
#     if "Focus" not in geo_data: return "" 
#     fn = geo_data['Focus']['Name']
#     fv_form = geo_data['Focus']['Formatted_Value']
#     fv_raw = geo_data['Focus'].get('Raw_Value')
#     f_cat = geo_data['Focus'].get('Category')
    
#     cy = years_ctx.get("latest", "Current Year")
#     py = years_ctx.get("prev", "Previous Year")
    
#     m_lower = metric_name.lower()
#     type_lower = str(m_type).lower()
#     is_yoy_change = ("change" in m_lower and "cumulative" not in m_lower)
    
#     if is_yoy_change:
#         context = f"Context: Compare the change between {py} and {cy}."
#     elif "cumulative" in m_lower:
#         context = f"Context: Compare the cumulative change from 1990 to {cy}."
#     else:
#         context = f"Context: The data is for the year {cy}."

#     is_percentage = False
#     if "percent" in type_lower or ("percent" in m_lower and "numeric" not in type_lower):
#         is_percentage = True
        
#     if is_percentage and abs(float(fv_raw or 0)) > 1000:
#         is_percentage = False
        
#     if is_percentage: type_group = "percentage"
#     elif "largest" in m_lower and "age" in m_lower: type_group = "ordinal"
#     elif is_categorical: type_group = "categorical"
#     else: type_group = "numerical"

#     display_metric_name = metric_name
#     display_desc = metric_desc
#     if not is_percentage:
#         display_metric_name = re.sub(r'\bpercentage\b|\bpercent\b', '', display_metric_name, flags=re.IGNORECASE).strip()
#         display_desc = re.sub(r'\bpercentage\b|\bpercent\b', '', display_desc, flags=re.IGNORECASE).strip()
#         display_metric_name = ' '.join(display_metric_name.split()) 

#     p = f"You are an expert data analyst writing for a professional dashboard. Metric: {display_metric_name}.\nDefinition: {display_desc}\n{context}\n"
#     p += "Write exactly ONE concise, professional sentence summarizing the data below.\n"
#     p += "CRITICAL RULES:\n"
#     p += "1. NO conversational filler.\n"
#     p += f"2. Start your sentence directly with the exact name: {fn}.\n"
#     p += "3. NEVER abbreviate geography names. Use the exact names provided in the data.\n"
#     p += "4. Output STRICTLY as a valid JSON object formatted as: { \"insight\": \"your sentence here\" }\n"

#     if is_yoy_change or "change" in m_lower:
#         p += "5. CRITICAL RULE: This metric represents a CHANGE or DIFFERENCE over time. You MUST phrase it as a change (e.g., 'increased by', 'decreased by', 'a change of'), NOT as the total absolute rate.\n"

#     p += "6. TONE/STYLE EXAMPLES TO FOLLOW:\n"
#     if type_group == "percentage":
#         p += "  - Internal Snapshot Example: { \"insight\": \"In 2026, 10.9% of Travis County residents lived below the poverty line.\" }\n"
#         p += "  - Internal Change Example: { \"insight\": \"Travis County saw a 0.8% decrease in its poverty rate from 2025 to 2026.\" }\n"
#         p += "  - Comparative Snapshot Example: { \"insight\": \"At 10.9% in 2026, Travis County's poverty rate sits 1.6 percentage points lower than the national average.\" }\n"
#         p += "  - Comparative Change Example: { \"insight\": \"Travis County's poverty rate decreased by 0.1 percentage points from 2025 to 2026, tracking lower than its peer average.\" }\n"
#         p += "  - CRITICAL VOCABULARY RULE: Do NOT use words like 'increase', 'decrease', 'rise', or 'fall' when comparing two different regions in the same year. Use 'higher' or 'lower' instead.\n"
#     elif type_group == "numerical":
#         p += "  - Internal Snapshot Example: { \"insight\": \"Travis County's population reached 1.39 million residents in 2026.\" }\n"
#         p += "  - Internal Change Example: { \"insight\": \"Between 2025 and 2026, Travis County experienced a population increase of 14,749 residents.\" }\n"
#         p += "  - Comparative Snapshot Example: { \"insight\": \"Travis County's 2026 population of 1.39 million accounts for roughly 53% of the broader Austin MSA.\" }\n"
#         p += "  - Comparative Change Example: { \"insight\": \"Travis County experienced a population increase of 14,749 residents, which is smaller than the Austin MSA's growth of 53,796.\" }\n"
#         p += "  - CRITICAL VOCABULARY RULE: Do NOT use competitive verbs like 'outpacing' or 'beating'. Use objective descriptions like 'larger', 'smaller', 'higher', or 'lower'.\n"
#     elif type_group == "categorical":
#         p += "  - Internal Snapshot Example: { \"insight\": \"In 2026, the largest demographic in Travis County was the White population, comprising 637,377 residents.\" }\n"
#         p += "  - Internal Change Example: { \"insight\": \"Travis County's largest driver of population change in 2026 was international migration, adding 11,928 residents.\" }\n"
#         p += "  - Comparative Snapshot Example: { \"insight\": \"While Travis County is predominantly White (637,377 residents), Texas statewide is primarily driven by its Hispanic population.\" }\n"
#         p += "  - Comparative Change Example: { \"insight\": \"In Travis County, international migration drove the largest population change (+11,928 residents), whereas the Austin MSA saw domestic migration as its primary contributor.\" }\n"
#         p += "  - CRITICAL VOCABULARY RULE: Do NOT use competitive verbs like 'outpacing', 'exceeding', or 'surpassing' when comparing different categories. Use neutral transitions like 'whereas' or 'while'.\n"
#     elif type_group == "ordinal":
#         p += "  - Internal Snapshot Example: { \"insight\": \"Travis County's largest age group in 2026 is the 30-to-34 cohort, comprising 272,962 individuals.\" }\n"
#         p += "  - Internal Change Example: { \"insight\": \"Between 2025 and 2026, Travis County saw its most significant demographic shift in the 75-to-79 age bracket.\" }\n"
#         p += "  - Comparative Snapshot Example: { \"insight\": \"Travis County's largest age group is the 30-to-34 cohort (272,962 individuals), which represents a smaller share compared to its combined peers.\" }\n"
#         p += "  - Comparative Change Example: { \"insight\": \"Travis County's recent demographic growth was driven by the 75-to-79 bracket, contrasting with the broader Austin MSA where the 40-to-44 cohort expanded most.\" }\n"
#         p += "  - CRITICAL VOCABULARY RULE: Do NOT use competitive verbs like 'outpacing', 'exceeding', or 'surpassing' when comparing different age groups. Use neutral transitions like 'whereas', 'while', or 'contrasting with'.\n"

#     if insight_type == "Internal": 
#         p += f"\nData:\n- Focus Geography ({fn}): {fv_form}\nTask: Analyze {fn} independently."
#     else:
#         comp_role = insight_type
#         if comp_role in geo_data:
#             cn_raw = geo_data[comp_role]['Name']
#             cn_clean = cn_raw.replace("Peers (Average)", "its peer average").replace("Peers (Combined)", "its combined peers")
#             cv_form = geo_data[comp_role]['Formatted_Value']
#             cv_raw = geo_data[comp_role].get('Raw_Value')
#             c_cat = geo_data[comp_role].get('Category')
            
#             math_hint = get_required_fact(fv_raw, cv_raw, cn_raw, metric_name, m_type, is_categorical, f_cat, c_cat)
#             if math_hint:
#                 p += f"7. {math_hint} You must integrate this fact naturally into your sentence. Do not perform your own math.\n"
                
#             p += f"\nData:\n- Focus Geography ({fn}): {fv_form}\n- Comparison Geography ({cn_clean}): {cv_form}\nTask: Compare them accurately based on the rules."
#         else: return ""
#     return p

# def main():
#     print("Loading data and Blueprint...")
#     df_acs = pl.read_csv("ACS_Series_Polars.csv", ignore_errors=True)
#     df_comp = pl.read_csv("components_of_change (4).csv", ignore_errors=True)
#     df_pyr = pl.read_csv("population_pyramid.csv", ignore_errors=True)
    
#     try: df_blueprint = pl.read_excel("../Metric Topics (DRAFT).xlsx", engine='openpyxl')
#     except Exception as e:
#         print(f"Failed to load Excel blueprint. Error: {e}")
#         return pd.DataFrame()

#     master_df = pl.concat([standardize_dataset(df_acs, "ACS"), standardize_dataset(df_comp, "COMPONENTS"), standardize_dataset(df_pyr, "POP_PYRAMID")])
    
#     final_results = []
#     total_processing_time = 0
#     metrics_processed = 0
    
#     print(f"Blueprint loaded. Iterating through defined metrics...\n")

#     for row in df_blueprint.iter_rows(named=True):
#         m_topic = row.get("Topic", "")
#         m_name = row.get("Metric", "")
#         m_type = row.get("Metric Type", "")
#         m_desc = row.get("Description", "")
#         d_source = row.get("Data Source", "")
#         v_json = row.get("Variables", "")
        
#         bp_comp = str(row.get("Comparison Period", ""))
#         bp_curr = str(row.get("Current Period", ""))
        
#         if not m_name: continue
        
#         start_time = time.time()
        
#         geo_data, years_ctx, is_cat = calculate_metric_data(master_df, df_pyr, m_name, v_json, m_type)
#         if not geo_data:
#             continue
            
#         focus_geo = geo_data['Focus']['Name']
#         broad_geo = geo_data.get('Broad', {}).get('Name', '')
#         bench_geo = geo_data.get('Benchmark', {}).get('Name', '')
#         peer_geo = geo_data.get('Peer', {}).get('Name', '')
        
#         # --- GENERATE + GRAMMAR + GEOGRAPHY STANDARDIZATION ---
#         i_int_raw = get_ollama_text(create_prompt("Internal", geo_data, m_name, m_desc, years_ctx, is_cat, m_type), as_json=True, json_key="insight")
#         i_int_grammar = apply_grammar_layer(i_int_raw)
#         i_int = apply_geography_standardization_layer(i_int_grammar, [focus_geo])
        
#         if broad_geo:
#             i_brd_raw = get_ollama_text(create_prompt("Broad", geo_data, m_name, m_desc, years_ctx, is_cat, m_type), as_json=True, json_key="insight")
#             i_brd_grammar = apply_grammar_layer(i_brd_raw)
#             i_brd = apply_geography_standardization_layer(i_brd_grammar, [focus_geo, broad_geo])
#         else: i_brd = "N/A"
        
#         if bench_geo:
#             i_bnc_raw = get_ollama_text(create_prompt("Benchmark", geo_data, m_name, m_desc, years_ctx, is_cat, m_type), as_json=True, json_key="insight")
#             i_bnc_grammar = apply_grammar_layer(i_bnc_raw)
#             i_bnc = apply_geography_standardization_layer(i_bnc_grammar, [focus_geo, bench_geo])
#         else: i_bnc = "N/A"
        
#         if peer_geo:
#             i_per_raw = get_ollama_text(create_prompt("Peer", geo_data, m_name, m_desc, years_ctx, is_cat, m_type), as_json=True, json_key="insight")
#             i_per_grammar = apply_grammar_layer(i_per_raw)
#             i_per = apply_geography_standardization_layer(i_per_grammar, [focus_geo, peer_geo])
#         else: i_per = "N/A"
        
#         cy = years_ctx.get("latest", "")
#         py = years_ctx.get("prev", "")
        
#         if cy:
#             bp_comp = bp_comp.replace("[Current Year]", cy)
#             bp_curr = bp_curr.replace("[Current Year]", cy)
#         if py:
#             bp_comp = bp_comp.replace("[Previous Year]", py)
#             bp_curr = bp_curr.replace("[Previous Year]", py)
            
#         # --- CHEAT SHEET + SYNTHESIS EXAMPLES ---
#         fv_focus = geo_data.get('Focus', {}).get('Formatted_Value', 'N/A')
#         fv_broad = geo_data.get('Broad', {}).get('Formatted_Value', 'N/A')
#         fv_bench = geo_data.get('Benchmark', {}).get('Formatted_Value', 'N/A')
#         fv_peer = geo_data.get('Peer', {}).get('Formatted_Value', 'N/A')

#         synth = f"""
#         You are a data analyst. Synthesize these 4 insights about {m_name} for {cy} into ONE coherent summary sentence.
        
#         REFERENCE VALUES (Use these to determine if {focus_geo} is higher/lower than the others):
#         - Focus ({focus_geo}): {fv_focus}
#         - Broad ({broad_geo}): {fv_broad}
#         - Benchmarks ({bench_geo}): {fv_bench}
#         - Peers ({peer_geo}): {fv_peer}

#         Insights to synthesize:
#         1. Internal: {i_int} 
#         2. Broad: {i_brd} 
#         3. Benchmarks: {i_bnc} 
#         4. Peers: {i_per}
        
#         CRITICAL RULES:
#         1. DO NOT merge different geographic names together. 
#         2. DO NOT compare a geography to itself.
#         3. Check the REFERENCE VALUES before stating if {focus_geo} is higher or lower than another region. Do NOT include the specific mathematical differences in this summary.
#         4. If comparing different demographic categories or age groups, DO NOT use words like 'outpacing' or 'exceeding'. Use neutral terms like 'whereas' or 'while'.
#         5. Start directly with "{focus_geo}".
        
#         EXAMPLE SYNTHESIS SENTENCES TO MIMIC:
#         - "Travis County's population reached 1.39 million residents in 2026, which is smaller than the Austin MSA but higher than its peer average."
#         - "Travis County experienced a 0.8% decrease in its poverty rate from 2025 to 2026, reflecting a lower rate than both the national average and its combined peers."
#         - "Travis County saw international migration drive its largest population change (+11,928 residents), whereas the Austin MSA and its combined peers were primarily driven by domestic migration."
        
#         Output STRICTLY as a valid JSON object: {{ "overall_insight": "your sentence here" }}
#         """
        
#         i_over_raw = get_ollama_text(synth, as_json=True, json_key="overall_insight") if i_int != "N/A" else "N/A"
#         i_over_grammar = apply_grammar_layer(i_over_raw)
#         i_over = apply_geography_standardization_layer(i_over_grammar, [focus_geo, broad_geo, bench_geo, peer_geo])
        
#         end_time = time.time()
#         processing_time = round(end_time - start_time, 2)
#         total_processing_time += processing_time
#         metrics_processed += 1
        
#         print(f"\n[✓] Processed Metric {metrics_processed}: '{m_name}' in {processing_time}s")
#         print(f"  [Internal]   {i_int}")
#         print(f"  [Broad]      {i_brd}")
#         print(f"  [Benchmarks] {i_bnc}")
#         print(f"  [Peers]      {i_per}")
#         print(f"  [Overall]    {i_over}")
#         print("-" * 75)
        
#         final_results.append({
#             "Topic": m_topic,
#             "Comparison Period": bp_comp, 
#             "Current Period": bp_curr,    
#             "Data Source": d_source,
#             "Variables": v_json,
#             "Metric": m_name,
#             "Metric Type": m_type,
#             "Description": m_desc,
#             "Internal Insight": i_int,
#             "Comparative Insight (Broad)": i_brd,
#             "Comparative Insight (Benchmarks)": i_bnc,
#             "Comparative Insight (Peers)": i_per,
#             "Overall Insight": i_over
#         })

#     if metrics_processed == 0: return pd.DataFrame()

#     columns_ordered = [
#         "Topic", "Comparison Period", "Current Period", 
#         "Data Source", "Variables", "Metric", "Metric Type", "Description", 
#         "Internal Insight", "Comparative Insight (Broad)", 
#         "Comparative Insight (Benchmarks)", "Comparative Insight (Peers)", 
#         "Overall Insight"
#     ]
#     df_final = pd.DataFrame(final_results)[columns_ordered]
    
#     avg_time = round(total_processing_time / metrics_processed, 2)
#     print("\n" + "="*50)
#     print(f"PIPELINE COMPLETE: {metrics_processed} Metrics processed.")
#     print(f"Average Processing Time: {avg_time} seconds/metric")
#     print("="*50 + "\n")
    
#     df_final.to_csv("dashboard_data_debug.csv", index=False)
#     return df_final

# if __name__ == "__main__":
#     df = main()

In [22]:
import polars as pl
import pandas as pd
import ollama
import time
import json
import random 
import re 

MODEL_NAME = "gemma3" 

# ==========================================
# 1. STANDARDIZATION & FORMATTING
# ==========================================
def standardize_dataset(df: pl.DataFrame, source_type: str) -> pl.DataFrame:
    # Aggressively strip column names to prevent missed lookups
    df.columns = [c.strip() for c in df.columns]
    
    # Adaptable geography standardization: Rely on 'Display Name' if available, otherwise 'NAME'
    if "Display Name" in df.columns:
        df = df.with_columns(
            pl.when(pl.col("Display Name").is_not_null() & (pl.col("Display Name").cast(pl.Utf8).str.strip_chars() != ""))
            .then(pl.col("Display Name").cast(pl.Utf8).str.strip_chars())
            .otherwise(pl.col("NAME"))
            .alias("NAME")
        )
        
    # # Global string standardization to ensure MSAs and Cities match cleanly
    # df = df.with_columns(
    #     pl.col("NAME")
    #     .str.replace(" city, Texas", " City")
    #     .str.replace(" city, Colorado", ", CO")
    #     .str.replace(", Texas", "")
    #     .str.replace(" Metro Area", "")
    #     .str.replace("-Round Rock-San Marcos, TX", "")
    #     .str.replace("-New Braunfels, TX", "")
    #     .str.replace("-Fort Worth-Arlington, TX", "")
    #     .str.replace("-Pasadena-The Woodlands, TX", "")
    #     .str.strip_chars()
    # )

    if source_type == "ACS":
        return df.select([pl.col("Year").cast(pl.Int64), pl.col("Cleaned Field Name").alias("Metric"), pl.col("NAME"), pl.col("Value").cast(pl.Float64), pl.col("Role")])
    elif source_type == "COMPONENTS":
        return df.select([pl.col("Year").cast(pl.Int64), pl.col("Value Type").alias("Metric"), pl.col("NAME"), pl.col("Values").alias("Value").cast(pl.Float64), pl.col("Role")])
    elif source_type == "POP_PYRAMID":
        return df.select([pl.col("YEAR").cast(pl.Int64).alias("Year"), (pl.col("Variable Group Description") + " (" + pl.col("Age Group Description") + ")").alias("Metric"), pl.col("NAME"), pl.col("Values").alias("Value").cast(pl.Float64), pl.col("Role")])

def format_value(value, metric_name, m_type=""):
    try:
        val = float(value)
        if val != val: return "N/A"
    except: return str(value)
    
    m_lower = str(metric_name).lower()
    type_lower = str(m_type).lower()
    
    is_percentage = False
    if "percent" in type_lower or "rate" in type_lower:
        is_percentage = True
    elif ("percent" in m_lower or "rate" in m_lower) and "numeric" not in type_lower:
        is_percentage = True
        
    if is_percentage and abs(val) > 1000:
        is_percentage = False
    
    if is_percentage:
        return f"{val:.1f}%"
            
    elif "age" in m_lower and "largest" not in m_lower:  
        if val == 0: return "0.0"
        return f"+{val:.1f}" if ("change" in m_lower and val > 0) else f"{val:.1f}"
    elif "income" in m_lower or "cost" in m_lower: 
        return f"${val:,.0f}"
    elif "change" in m_lower or "net" in m_lower or "mig" in m_lower or "driver" in m_lower: 
        if val == 0: return "0"
        return f"+{val:,.0f}" if val > 0 else f"{val:,.0f}"
    else: 
        if val >= 1_000_000: return f"{val/1_000_000:.2f} million"
        return f"{val:,.0f}"

# ==========================================
# 2. REQUIRED FACT GENERATOR (Math Lock)
# ==========================================
def get_required_fact(fv_raw, cv_raw, fn, cn, metric_name, m_type, is_categorical=False, f_cat=None, c_cat=None):
    """Generates strict mathematical instructions to prevent LLM hallucination or metric swapping."""
    try:
        f = float(fv_raw)
        c = float(cv_raw)
        if c == 0 and f != 0: return ""
        
        m_lower = str(metric_name).lower()
        type_lower = str(m_type).lower()
        cn_first = cn.replace("Peers (Average)", "its peer average").replace("Peers (Combined)", "its combined peers").split(" and ")[0] 
        
        is_percentage = False
        if "percent" in type_lower or "rate" in type_lower:
            is_percentage = True
        elif ("percent" in m_lower or "rate" in m_lower) and "numeric" not in type_lower:
            is_percentage = True
            
        if is_percentage and (abs(f) > 1000 or abs(c) > 1000):
            is_percentage = False

        f_str = format_value(f, metric_name, m_type)
        c_str = format_value(c, metric_name, m_type)

        base_lock = f"MATH LOCK: {fn} is exactly {f_str} and {cn_first} is exactly {c_str}. DO NOT confuse or swap these values."

        if is_categorical:
            if f_cat == c_cat:
                diff = f - c
                return f"{base_lock} REQUIRED FACT: You must state both share the primary group '{f_cat}', but the magnitude differs by {abs(diff):,.0f}."
            else:
                return f"{base_lock} REQUIRED FACT: You must explicitly state that the primary group is '{f_cat}', whereas for {cn_first} it is '{c_cat}'. DO NOT subtract their values."
        
        if f == c:
            return f"{base_lock} REQUIRED FACT: You must explicitly state that the values are exactly identical. DO NOT say it is higher or lower."
            
        if m_lower in ["median age", "change in median age"]:
            diff = f - c
            return f"{base_lock} REQUIRED FACT: You must state the difference is exactly {abs(diff):.1f} years {'older' if diff > 0 else 'younger'} than {cn_first}."
        elif is_percentage:
            diff = f - c
            return f"{base_lock} REQUIRED FACT: You must state the difference is exactly {abs(diff):.1f} percentage points {'higher' if diff > 0 else 'lower'} than {cn_first}. DO NOT use the '%' symbol for the difference."
        elif "change" in m_lower or "net" in m_lower or "mig" in m_lower or "driver" in m_lower:
            diff = f - c
            return f"{base_lock} REQUIRED FACT: You must state the difference is exactly {abs(diff):,.0f} {'more' if diff > 0 else 'fewer'} than {cn_first}."
        else:
            ratio = f / c
            if ratio < 1: return f"{base_lock} REQUIRED FACT: You must state it is approximately {ratio*100:.1f}% the size of {cn_first}."
            else: return f"{base_lock} REQUIRED FACT: You must state it is approximately {ratio:.1f} times larger than {cn_first}."
    except:
        return ""

def get_categorical_peer_detailed_draft(f_val, f_cat, peer_details, fn, metric_name, m_type):
    """Calculates categorical qualitative relationships and returns top 5 largest peers."""
    try:
        f_str = format_value(f_val, metric_name, m_type)
        same_cat = []
        diff_cat = []
        
        debug_math = f"Focus ({fn}) Cat: {f_cat} ({f_str}) | "
        
        for p in peer_details:
            if fn.lower() == p["Name"].lower() or p["Name"].lower() in fn.lower():
                continue
            p_val = float(p["Value"])
            p_str = format_value(p_val, metric_name, m_type)
            peer_obj = {"name": p["Name"], "cat": p["Category"], "val": p_val, "str": p_str}
            
            debug_math += f"[{p['Name']}: {p['Category']} ({p_str})] "
            if p["Category"] == f_cat:
                same_cat.append(peer_obj)
            else:
                diff_cat.append(peer_obj)
                
        # Sort by actual numerical size descending to rank importance
        same_cat.sort(key=lambda x: x["val"], reverse=True)
        diff_cat.sort(key=lambda x: x["val"], reverse=True)
        
        def fmt_cat_peers(lst):
            if not lst: return []
            return [f"{x['name']} ({x['cat']} at {x['str']})" for x in lst]
            
        same_names = fmt_cat_peers(same_cat)
        diff_names = fmt_cat_peers(diff_cat)
                
        draft = f"{fn}'s primary {metric_name.lower()} is {f_cat} ({f_str}). "
        
        if same_names:
            if len(same_names) > 5:
                draft += f"It shares this primary category with {', '.join(same_names[:5])}, and {len(same_names)-5} others. "
            elif len(same_names) == 1: 
                draft += f"It shares this primary category with {same_names[0]}. "
            elif len(same_names) == 2: 
                draft += f"It shares this primary category with {same_names[0]} and {same_names[1]}. "
            else: 
                draft += f"It shares this primary category with {', '.join(same_names[:-1])}, and {same_names[-1]}. "
                
        if diff_names:
            if len(diff_names) > 5:
                draft += f"In contrast, the primary category differs for {', '.join(diff_names[:5])}, and {len(diff_names)-5} others. "
            elif len(diff_names) == 1: 
                draft += f"In contrast, the primary category differs for {diff_names[0]}. "
            elif len(diff_names) == 2: 
                draft += f"In contrast, the primary category differs for {diff_names[0]} and {diff_names[1]}. "
            else: 
                draft += f"In contrast, the primary category differs for {', '.join(diff_names[:-1])}, and {diff_names[-1]}. "
            
        return draft, debug_math
    except Exception as e:
        return "", f"Error: {str(e)}"

def get_peer_detailed_draft(fv_raw, peer_details, fn, metric_name, is_percentage=False, m_type=""):
    """Calculates numerical comparative relationships and returns top 5 by absolute difference."""
    if not peer_details or fv_raw is None: return "", "Missing Data"
    try:
        f = float(fv_raw)
        higher_than = []
        lower_than = []
        similar = []
        
        if is_percentage:
            tolerance = max(abs(f * 0.02), 0.2)
        else:
            tolerance = max(abs(f * 0.02), 0.001)
            
        debug_math = f"Focus ({fn}): {f:,.4f} | Tol: +/-{tolerance:,.4f} | "
        f_str = format_value(f, metric_name, m_type)
        
        for p in peer_details:
            if fn.lower() == p["Name"].lower() or p["Name"].lower() in fn.lower():
                continue
                
            p_val = float(p["Value"])
            p_str = format_value(p_val, metric_name, m_type)
            diff = f - p_val
            
            peer_obj = {"name": p["Name"], "val": p_val, "str": p_str, "abs_diff": abs(diff)}
            
            debug_math += f"[{p['Name']} ({p_val:,.4f}): "
            if abs(diff) <= tolerance:
                similar.append(peer_obj)
                debug_math += "SIM] "
            elif diff > 0:
                higher_than.append(peer_obj)
                debug_math += "Focus>Peer] "
            else:
                lower_than.append(peer_obj)
                debug_math += "Focus<Peer] "
                
        # Sort lists by highest absolute difference
        higher_than.sort(key=lambda x: x["abs_diff"], reverse=True)
        lower_than.sort(key=lambda x: x["abs_diff"], reverse=True)
        similar.sort(key=lambda x: x["abs_diff"], reverse=True)

        metric_str = metric_name.lower()
        if is_percentage or "rate" in metric_str or "change" in metric_str:
            word_high = "higher"
            word_low = "lower"
        else:
            word_high = "larger"
            word_low = "smaller"
            
        draft = f"{fn}'s {metric_str} of {f_str} is evaluated against individual peers. "
            
        def fmt_list(lst):
            if not lst: return ""
            names = [f"{x['name']} ({x['str']})" for x in lst]
            if len(names) > 5:
                return ", ".join(names[:5]) + f", and {len(names)-5} others"
            if len(names) == 1: return names[0]
            if len(names) == 2: return f"{names[0]} and {names[1]}"
            return ", ".join(names[:-1]) + f", and {names[-1]}"
            
        parts = []
        if lower_than: parts.append(f"{word_low} than {fmt_list(lower_than)}")
        if higher_than: parts.append(f"{word_high} than {fmt_list(higher_than)}")
        if similar: parts.append(f"nearly the same as {fmt_list(similar)}")
        
        if parts:
            draft += "Specifically, it is " + ", while being ".join(parts) + "."
        else:
            draft += "It is not comparable to specific peers."
            
        return draft, debug_math
    except Exception as e:
        return "", f"Error: {str(e)}"

# ==========================================
# 3. UNIFIED DATA ENGINE
# ==========================================
def combine_roles(df_result, value_col, metric_name, m_type, is_categorical=False, category_col=None):
    geo_data = {}
    if not is_categorical:
        df_result = df_result.sort(value_col) 
        
    role_dict = {}
    for r in df_result.filter(~pl.col("Role").is_in(["Peer", "Peer_Indiv"])).iter_rows(named=True):
        role = r["Role"]
        if role not in role_dict: role_dict[role] = []
        role_dict[role].append(r)
            
    for role, rows in role_dict.items():
        if len(rows) == 1:
            r = rows[0]
            if is_categorical: 
                geo_data[role] = {"Name": r["NAME"], "Formatted_Value": f"the '{r[category_col]}' group with {format_value(r[value_col], metric_name, m_type)}", "Raw_Value": r[value_col], "Category": r[category_col]}
            else: 
                geo_data[role] = {"Name": r["NAME"], "Formatted_Value": format_value(r[value_col], metric_name, m_type), "Raw_Value": r[value_col]}
        else:
            combined_name = " and ".join([r["NAME"] for r in rows])
            if is_categorical: 
                combined_formatted = " and ".join([f"{r['NAME']}: the '{r[category_col]}' group with {format_value(r[value_col], metric_name, m_type)}" for r in rows])
                geo_data[role] = {"Name": combined_name, "Formatted_Value": combined_formatted, "Raw_Value": rows[0][value_col], "Category": rows[0][category_col]}
            else: 
                combined_formatted = " and ".join([f"{r['NAME']}: {format_value(r[value_col], metric_name, m_type)}" for r in rows])
                geo_data[role] = {"Name": combined_name, "Formatted_Value": combined_formatted, "Raw_Value": rows[0][value_col]}
            
    df_peer = df_result.filter(pl.col("Role") == "Peer")
    if not df_peer.is_empty():
        if is_categorical:
            r = df_peer.row(0, named=True)
            geo_data["Peer"] = {"Name": r["NAME"], "Formatted_Value": f"the '{r[category_col]}' group with {format_value(r[value_col], metric_name, m_type)}", "Raw_Value": r[value_col], "Category": r[category_col]}
        else:
            peer_avg = df_peer[value_col].mean()
            geo_data["Peer"] = {"Name": "Peers (Average)", "Formatted_Value": format_value(peer_avg, metric_name, m_type), "Raw_Value": peer_avg}
            
            peer_details = []
            for r in df_peer.iter_rows(named=True):
                peer_details.append({"Name": r["NAME"], "Value": r[value_col]})
            geo_data["Peer_Details"] = peer_details

    df_peer_indiv = df_result.filter(pl.col("Role") == "Peer_Indiv")
    if not df_peer_indiv.is_empty():
        peer_details = []
        for r in df_peer_indiv.iter_rows(named=True):
            cat = r[category_col] if is_categorical else None
            peer_details.append({"Name": r["NAME"], "Value": r[value_col], "Category": cat})
        geo_data["Peer_Details"] = peer_details
            
    return geo_data

def calculate_metric_data(master_df, df_pyr, metric_name, variables_json, m_type):
    m_lower = metric_name.lower()
    type_lower = str(m_type).lower()
    
    is_yoy_change = ("change" in m_lower and "cumulative" not in m_lower)
    is_percentage_metric = "percent" in type_lower or "rate" in type_lower or (("percent" in m_lower or "rate" in m_lower) and "numeric" not in type_lower)
    
    years_context = {"latest": "", "prev": ""}
    
    if "driver" in m_lower or "dynamics in drivers" in m_lower:
        is_cumulative = "cumulative" in m_lower
        drivers = ["NATURALCHG", "DOMESTICMIG", "INTERNATIONALMIG"]
        df_d = master_df.filter(pl.col("Metric").is_in(drivers))
        
        if df_d.is_empty(): return None, years_context, True
        
        years = df_d["Year"].drop_nulls().unique().sort()
        if len(years) > 0: years_context["latest"] = str(years[-1])
        if len(years) > 1: years_context["prev"] = str(years[-2])
        
        if is_cumulative: df_d = df_d.filter(pl.col("Year") >= 1990)
        else: df_d = df_d.filter(pl.col("Year") == df_d["Year"].max())
        if df_d.is_empty(): return None, years_context, True
        
        df_std = df_d.filter(pl.col("Role") != "Peer")
        
        df_peer_indiv = df_d.filter(pl.col("Role") == "Peer")
        df_peer_comb = df_peer_indiv.with_columns(pl.lit("Peers (Combined)").alias("NAME"))
        df_combined = pl.concat([df_std, df_peer_comb, df_peer_indiv.with_columns(pl.lit("Peer_Indiv").alias("Role"))])
        
        agg = df_combined.group_by(["NAME", "Role", "Metric"]).agg(pl.col("Value").sum())
        largest = agg.with_columns(pl.col("Value").abs().alias("Abs_Val")).sort("Abs_Val", descending=True).group_by(["NAME", "Role"]).first()
        
        names = {"NATURALCHG": "Natural Change", "DOMESTICMIG": "Domestic Migration", "INTERNATIONALMIG": "International Migration"}
        largest = largest.with_columns(pl.col("Metric").replace(names).alias("Driver_Name"))
        
        return combine_roles(largest, "Value", metric_name, m_type, is_categorical=True, category_col="Driver_Name"), years_context, True

    elif "cumulative" in m_lower and "population" in m_lower and "change" in m_lower:
        df_pop = master_df.filter(pl.col("Metric") == "POPESTIMATE")
        if df_pop.is_empty() or df_pop.filter(pl.col("Year") == 1990).is_empty(): return None, years_context, False
        
        years = df_pop["Year"].drop_nulls().unique().sort()
        if len(years) > 0: years_context["latest"] = str(years[-1])
        if len(years) > 1: years_context["prev"] = str(years[-2])
        
        latest_year = df_pop["Year"].max()
        df_latest = df_pop.filter(pl.col("Year") == latest_year).select(["NAME", "Role", "Value"])
        df_1990 = df_pop.filter(pl.col("Year") == 1990).select(["NAME", "Role", "Value"])
        
        change_df = df_latest.join(df_1990, on=["NAME", "Role"], how="inner")
        
        if is_percentage_metric:
            change_df = change_df.with_columns((((pl.col("Value") - pl.col("Value_right")) / pl.col("Value_right")) * 100).alias("Change"))
        else:
            change_df = change_df.with_columns((pl.col("Value") - pl.col("Value_right")).alias("Change"))
            
        return combine_roles(change_df, "Change", metric_name, m_type), years_context, False

    elif "largest" in m_lower and ("age" in m_lower or "race" in m_lower):
        years = df_pyr["YEAR"].drop_nulls().unique().sort()
        if len(years) == 0: return None, years_context, True
        
        latest = years[-1]
        years_context["latest"] = str(latest)
        if len(years) > 1: years_context["prev"] = str(years[-2])
        
        if "race" in m_lower:
            df_pyr_clean = df_pyr.with_columns(pl.col("Variable Group Description").str.replace(" Male", "").str.replace(" Female", "").alias("Group_Clean"))
            df_pyr_clean = df_pyr_clean.filter(~pl.col("Group_Clean").is_in(["Male", "Female", "Total", "All"]))
            group_col = "Group_Clean"
        else:
            df_pyr_clean = df_pyr.filter(pl.col("Age Group Description") != "All")
            group_col = "Age Group Description"
            
        df_std = df_pyr_clean.filter(pl.col("Role") != "Peer")
        df_peer_indiv = df_pyr_clean.filter(pl.col("Role") == "Peer")
        df_peer_comb = df_peer_indiv.with_columns(pl.lit("Peers (Combined)").alias("NAME"))
        df_combined = pl.concat([df_std, df_peer_comb, df_peer_indiv.with_columns(pl.lit("Peer_Indiv").alias("Role"))])

        if not is_yoy_change:
            df_l = df_combined.filter(pl.col("YEAR") == latest).group_by(["NAME", "Role", group_col]).agg(pl.col("Values").cast(pl.Float64).sum())
            largest = df_l.sort("Values", descending=True).group_by(["NAME", "Role"]).first()
            return combine_roles(largest, "Values", metric_name, m_type, is_categorical=True, category_col=group_col), years_context, True
        else:
            if len(years) < 2: return None, years_context, True
            prev = years[-2] 
            
            df_l = df_combined.filter(pl.col("YEAR") == latest).group_by(["NAME", "Role", group_col]).agg(pl.col("Values").cast(pl.Float64).sum().alias("VL"))
            df_p = df_combined.filter(pl.col("YEAR") == prev).group_by(["NAME", "Role", group_col]).agg(pl.col("Values").cast(pl.Float64).sum().alias("VP"))
            change = df_l.join(df_p, on=["NAME", "Role", group_col], how="inner").with_columns((pl.col("VL") - pl.col("VP")).alias("Chg"))
            largest = change.sort("Chg", descending=True).group_by(["NAME", "Role"]).first()
            return combine_roles(largest, "Chg", metric_name, m_type, is_categorical=True, category_col=group_col), years_context, True

    else:
        target_raw = metric_name 
        if variables_json and isinstance(variables_json, str):
            try: target_raw = list(json.loads(variables_json.replace('""', '"')).values())[0] 
            except: pass
                
        df_metric = master_df.filter(pl.col("Metric") == target_raw)
        if df_metric.is_empty(): return None, years_context, False
            
        if is_percentage_metric:
            abs_max = df_metric.select(pl.col("Value").abs().max()).item()
            if abs_max is not None and abs_max <= 1.05 and abs_max > 0:
                df_metric = df_metric.with_columns(pl.col("Value") * 100)
            
        years = df_metric["Year"].drop_nulls().unique().sort()
        if len(years) > 0: years_context["latest"] = str(years[-1])
        if len(years) > 1: years_context["prev"] = str(years[-2])
            
        latest_year = df_metric["Year"].max()
        
        if is_yoy_change:
            if len(years) < 2: return None, years_context, False
            prev_year = years[-2]
            
            df_l_dedup = df_metric.filter(pl.col("Year") == latest_year).group_by(["NAME", "Role"]).agg(pl.col("Value").mean())
            df_p_dedup = df_metric.filter(pl.col("Year") == prev_year).group_by(["NAME", "Role"]).agg(pl.col("Value").mean())
            change_df = df_l_dedup.join(df_p_dedup, on=["NAME", "Role"], how="inner")
            
            max_val = change_df["Value_right"].max()
            if is_percentage_metric and max_val and max_val > 100:
                change_df = change_df.with_columns((((pl.col("Value") - pl.col("Value_right")) / pl.col("Value_right")) * 100).alias("Change"))
            else:
                change_df = change_df.with_columns((pl.col("Value") - pl.col("Value_right")).alias("Change"))
                
            return combine_roles(change_df, "Change", metric_name, m_type), years_context, False
        else:
            df_dedup = df_metric.filter(pl.col("Year") == latest_year).group_by(["NAME", "Role"]).agg(pl.col("Value").mean())
            return combine_roles(df_dedup, "Value", metric_name, m_type), years_context, False


# ==========================================
# 4. LLM INTERFACE & PROMPTING
# ==========================================
def get_ollama_text(prompt, as_json=False, json_key="overall_insight"):
    if not prompt: return "N/A"
    
    # Progressively increase the prediction limits if the LLM output is truncated
    limits = [800, 1500, 2500]
    last_error = "Unknown Error"
    
    for limit in limits:
        try:
            resp = ollama.generate(
                model=MODEL_NAME, 
                prompt=prompt, 
                options={
                    "temperature": 0.1, 
                    "seed": random.randint(1, 100000),
                    "num_predict": limit
                }
            )['response'].strip()
            
            # Aggressive markdown and JSON artifact stripping
            resp_clean = re.sub(r'```json\s*', '', resp, flags=re.IGNORECASE)
            resp_clean = re.sub(r'```\s*', '', resp_clean)
            
            if as_json:
                match = re.search(r'\{.*?\}', resp_clean, re.DOTALL)
                if match:
                    try:
                        val = json.loads(match.group(0)).get(json_key, "")
                        if not val:
                            val = match.group(0)
                    except:
                        val = match.group(0)
                else:
                    val = resp_clean
                    
                # Deep fallback to wipe out keys if JSON completely shattered
                val = re.sub(r'\{?\s*\"?' + json_key + r'\"?\s*:\s*\"?', '', val, flags=re.IGNORECASE)
                val = val.replace('"}', '').replace('}', '').strip()
                val = re.sub(r'^\"', '', val)
                val = re.sub(r'\"$', '', val)
                val = re.sub(r"^.*?(revised_sentence|overall_insight|insight|topic_summary|complete_summary)\"?\s*:\s*\"?", "", val, flags=re.IGNORECASE).strip()
                val = re.sub(r"^.*?(summary|sentence|data|revised|professional).*?:", "", val, flags=re.IGNORECASE).strip()
                
                if not val:
                    last_error = "Empty parsed value"
                    continue # Retry on next limit
                
                # Check for severe truncation (If it doesn't end with typical punctuation)
                if not re.search(r'[.!?\"\'\}]$', val) and limit != limits[-1]:
                    last_error = "Truncated text detected (no ending punctuation)"
                    continue # Try again with a higher limit
                    
                return val.strip()
            else:
                return resp_clean.strip()
                
        except Exception as e: 
            last_error = str(e)
            continue
            
    return f"Error: {last_error}"

# ==========================================
# 5. POST-PROCESSING (Grammar & Geography)
# ==========================================
def apply_grammar_layer(text):
    if text in ["N/A", "JSON Error", "JSON Parse Error", "JSON Format Error", ""] or str(text).startswith("Error:") or str(text).startswith("Regex Error:"): return text
    
    p = f"""You are a strict copyeditor. Fix capitalization and grammar in the sentence below.
CRITICAL RULES:
1. Demographic categories, age groups, and drivers of change MUST be lowercase (e.g., 'international migration', 'domestic migration', 'natural change', 'white', 'hispanic') unless they start a sentence.
2. Geographic locations MUST be fully capitalized (e.g., 'Texas', 'United States', 'Austin MSA'). Do NOT lowercase them.
3. Output STRICTLY as a valid JSON object. Do not include intro text.

Original Sentence: {text}
Output format: {{ "revised_sentence": "your fixed sentence here" }}
"""
    return get_ollama_text(p, as_json=True, json_key="revised_sentence")

def apply_geography_standardization_layer(text, valid_geos):
    if text in ["N/A", "JSON Error", "JSON Parse Error", "JSON Format Error", ""] or str(text).startswith("Error:") or str(text).startswith("Regex Error:"): return text
    
    valid_geos_clean = list(set([g.replace("Peers (Average)", "its peer average").replace("Peers (Combined)", "its combined peers") for g in valid_geos if g]))
    geo_str = ", ".join([f"'{g}'" for g in valid_geos_clean])
    
    p = f"""You are a strict copyeditor. Ensure the geography names in the sentence below EXACTLY match the provided allowed list.
CRITICAL RULES:
1. The valid geography names for this sentence are EXACTLY: {geo_str}.
2. Replace any ALL-CAPS names (like 'TEXAS' or 'UNITED STATES') with their proper Title Case format as shown in the valid list.
3. Replace any unauthorized abbreviations (like 'Austin MSA') with the full name from the valid list, if applicable.
4. VERY IMPORTANT: Do NOT change one distinct geography into another (e.g., do not replace "Texas" with "Travis County, Texas"). Treat overlapping names as distinct entities.
5. Output STRICTLY as a valid JSON object. Do not include intro text.

Original Sentence: {text}
Output format: {{ "revised_sentence": "your fixed sentence here" }}
"""
    return get_ollama_text(p, as_json=True, json_key="revised_sentence")

# ==========================================
# 6. PIPELINE EXECUTION
# ==========================================
def create_prompt(insight_type, geo_data, metric_name, metric_desc, years_ctx, is_categorical, m_type):
    if "Focus" not in geo_data: return "", "" 
    fn = geo_data['Focus']['Name']
    fv_form = geo_data['Focus']['Formatted_Value']
    fv_raw = geo_data['Focus'].get('Raw_Value')
    f_cat = geo_data['Focus'].get('Category')
    
    cy = years_ctx.get("latest", "Current Year")
    py = years_ctx.get("prev", "Previous Year")
    
    m_lower = metric_name.lower()
    type_lower = str(m_type).lower()
    is_yoy_change = ("change" in m_lower and "cumulative" not in m_lower)
    
    if is_yoy_change:
        context = f"Context: Compare the change between {py} and {cy}."
    elif "cumulative" in m_lower:
        context = f"Context: Compare the cumulative change from 1990 to {cy}."
    else:
        context = f"Context: The data is for the year {cy}."

    is_percentage = False
    if "percent" in type_lower or ("percent" in m_lower and "numeric" not in type_lower):
        is_percentage = True
        
    if is_percentage and abs(float(fv_raw or 0)) > 1000:
        is_percentage = False
        
    if is_percentage: type_group = "percentage"
    elif "largest" in m_lower and "age" in m_lower: type_group = "ordinal"
    elif is_categorical: type_group = "categorical"
    else: type_group = "numerical"

    display_metric_name = metric_name
    display_desc = metric_desc
    if not is_percentage:
        display_metric_name = re.sub(r'\bpercentage\b|\bpercent\b', '', display_metric_name, flags=re.IGNORECASE).strip()
        display_desc = re.sub(r'\bpercentage\b|\bpercent\b', '', display_desc, flags=re.IGNORECASE).strip()
        display_metric_name = ' '.join(display_metric_name.split()) 

    if insight_type == "Peer_Detailed":
        peer_details = geo_data.get('Peer_Details', [])
        
        if is_categorical:
            draft, debug_math = get_categorical_peer_detailed_draft(fv_raw, f_cat, peer_details, fn, display_metric_name, m_type)
        else:
            draft, debug_math = get_peer_detailed_draft(fv_raw, peer_details, fn, display_metric_name, is_percentage, m_type)
            
        if not draft: return "", ""
        
        p = f"You are a professional data analyst. Rewrite the following DRAFT SENTENCE into a single, polished, professional statement about the metric: '{display_metric_name}'.\n\n"
        p += f"DRAFT SENTENCE: {draft}\n\n"
        p += "CRITICAL RULES:\n"
        p += "1. Output STRICTLY as a valid JSON object formatted as: { \"insight\": \"your polished sentence here\" }\n"
        p += "2. YOU MUST INCLUDE the exact numerical values or category names for EVERY geography exactly as they appear in the draft. Do NOT add trailing zeroes (e.g. .00) unless they are explicitly in the draft.\n"
        p += "3. DO NOT CONTRADICT THE DRAFT. Keep all larger/smaller/similar relationships exactly as stated.\n"
        p += "4. Keep the geographic relationships grouped logically. Do not try to reverse them or merge them inaccurately for stylistic variety.\n"
        p += f"5. NEVER compare '{fn}' to '{fn}'.\n"
        
        if is_categorical:
            p += f"6. Your sentence MUST flow naturally. Example: '{fn}'s primary {display_metric_name.lower()} is X, mirroring Peer A and Peer B, while Peer C differs with Y.'\n"
        else:
            p += f"6. Your sentence MUST flow naturally. Example: '{fn}'s {display_metric_name} of X is evaluated against its peers. Specifically, it is higher than Peer A (Z) and Peer B (W), but lower than Peer C (V).'\n"
            
        return p, debug_math

    p = f"You are an expert data analyst writing for a professional dashboard. Metric: {display_metric_name}.\nDefinition: {display_desc}\n{context}\n"
    p += "Write exactly ONE concise, professional sentence summarizing the data below.\n"
    p += "CRITICAL RULES:\n"
    p += "1. NO conversational filler.\n"
    p += f"2. Start your sentence directly with the exact name: {fn}.\n"
    p += "3. NEVER abbreviate geography names. Use the exact names provided in the data.\n"
    p += "4. Output STRICTLY as a valid JSON object formatted as: { \"insight\": \"your sentence here\" }\n"

    if is_yoy_change or "change" in m_lower:
        p += "5. CRITICAL RULE: This metric represents a CHANGE or DIFFERENCE over time. You MUST phrase it as a change (e.g., 'increased by', 'decreased by', 'a change of'), NOT as the total absolute rate.\n"

    p += "6. TONE/STYLE EXAMPLES TO FOLLOW:\n"
    if type_group == "percentage":
        p += "  - Internal Snapshot Example: { \"insight\": \"In 2026, 10.9% of Travis County residents lived below the poverty line.\" }\n"
        p += "  - Internal Change Example: { \"insight\": \"Travis County saw a 0.8% decrease in its poverty rate from 2025 to 2026.\" }\n"
        p += "  - Comparative Snapshot Example: { \"insight\": \"At 10.9% in 2026, Travis County's poverty rate sits 1.6 percentage points lower than the national average.\" }\n"
        p += "  - Comparative Change Example: { \"insight\": \"Travis County's poverty rate decreased by 0.1 percentage points from 2025 to 2026, tracking lower than its peer average.\" }\n"
        p += "  - CRITICAL VOCABULARY RULE: Do NOT use words like 'increase', 'decrease', 'rise', or 'fall' when comparing two different regions in the same year. Use 'higher' or 'lower' instead.\n"
    elif type_group == "numerical":
        p += "  - Internal Snapshot Example: { \"insight\": \"Travis County's population reached 1.39 million residents in 2026.\" }\n"
        p += "  - Internal Change Example: { \"insight\": \"Between 2025 and 2026, Travis County experienced a population increase of 14,749 residents.\" }\n"
        p += "  - Comparative Snapshot Example: { \"insight\": \"Travis County's 2026 population of 1.39 million accounts for roughly 53% of the broader Austin MSA.\" }\n"
        p += "  - Comparative Change Example: { \"insight\": \"Travis County experienced a population increase of 14,749 residents, which is smaller than the Austin MSA's growth of 53,796.\" }\n"
        p += "  - CRITICAL VOCABULARY RULE: Do NOT use competitive verbs like 'outpacing' or 'beating'. Use objective descriptions like 'larger', 'smaller', 'higher', or 'lower'.\n"
    elif type_group == "categorical":
        p += "  - Internal Snapshot Example: { \"insight\": \"In 2026, the largest demographic in Travis County was the White population, comprising 637,377 residents.\" }\n"
        p += "  - Internal Change Example: { \"insight\": \"Travis County's largest driver of population change in 2026 was international migration, adding 11,928 residents.\" }\n"
        p += "  - Comparative Snapshot Example: { \"insight\": \"While Travis County is predominantly White (637,377 residents), Texas statewide is primarily driven by its Hispanic population.\" }\n"
        p += "  - Comparative Change Example: { \"insight\": \"In Travis County, international migration drove the largest population change (+11,928 residents), whereas the Austin MSA saw domestic migration as its primary contributor.\" }\n"
        p += "  - CRITICAL VOCABULARY RULE: Do NOT use competitive verbs like 'outpacing', 'exceeding', or 'surpassing' when comparing different categories. Use neutral transitions like 'whereas' or 'while'.\n"
    elif type_group == "ordinal":
        p += "  - Internal Snapshot Example: { \"insight\": \"Travis County's largest age group in 2026 is the 30-to-34 cohort, comprising 272,962 individuals.\" }\n"
        p += "  - Internal Change Example: { \"insight\": \"Between 2025 and 2026, Travis County saw its most significant demographic shift in the 75-to-79 age bracket.\" }\n"
        p += "  - Comparative Snapshot Example: { \"insight\": \"Travis County's largest age group is the 30-to-34 cohort (272,962 individuals), which represents a smaller share compared to its combined peers.\" }\n"
        p += "  - Comparative Change Example: { \"insight\": \"Travis County's recent demographic growth was driven by the 75-to-79 bracket, contrasting with the broader Austin MSA where the 40-to-44 cohort expanded most.\" }\n"
        p += "  - CRITICAL VOCABULARY RULE: Do NOT use competitive verbs like 'outpacing', 'exceeding', or 'surpassing' when comparing different age groups. Use neutral transitions like 'whereas', 'while', or 'contrasting with'.\n"

    math_hint_debug = ""
    
    if insight_type == "Internal": 
        p += f"\nDATA BINDING:\n- [Focus Entity] '{fn}' MUST be reported as {fv_form}.\nTask: Analyze {fn} independently."
    else:
        comp_role = insight_type
        if comp_role in geo_data:
            cn_raw = geo_data[comp_role]['Name']
            cn_clean = cn_raw.replace("Peers (Average)", "its peer average").replace("Peers (Combined)", "its combined peers")
            cv_form = geo_data[comp_role]['Formatted_Value']
            cv_raw = geo_data[comp_role].get('Raw_Value')
            c_cat = geo_data[comp_role].get('Category')
            
            p += f"\nDATA BINDING:\n- [Focus Entity] '{fn}' MUST be reported as {fv_form}.\n- [Comparison Entity] '{cn_clean}' MUST be reported as {cv_form}.\n"
            
            math_hint = get_required_fact(fv_raw, cv_raw, fn, cn_raw, metric_name, m_type, is_categorical, f_cat, c_cat)
            math_hint_debug = math_hint
            if math_hint:
                p += f"7. {math_hint} Focus your sentence on describing this relationship using words like 'larger than', 'smaller than', 'higher than', or 'lower than'. DO NOT just list the numbers.\n"
                
            p += f"\nTask: Compare them accurately based on the rules. Do not confuse the Focus Entity with the Comparison Entity."
        else: return "", ""
    return p, math_hint_debug

def main():
    print("Loading data and Blueprint...")
    df_acs = pl.read_csv("ACS_Series_Polars.csv", ignore_errors=True)
    df_comp = pl.read_csv("components_of_change (4).csv", ignore_errors=True)
    df_pyr = pl.read_csv("population_pyramid.csv", ignore_errors=True)
    
    try: df_blueprint = pl.read_excel("Metric Topics (DRAFT).xlsx", engine='openpyxl')
    except Exception as e:
        print(f"Failed to load Excel blueprint. Error: {e}")
        return pd.DataFrame()

    master_df = pl.concat([standardize_dataset(df_acs, "ACS"), standardize_dataset(df_comp, "COMPONENTS"), standardize_dataset(df_pyr, "POP_PYRAMID")])
    
    final_results = []
    total_processing_time = 0
    metrics_processed = 0
    
    print(f"Blueprint loaded. Iterating through defined metrics...\n")

    for row in df_blueprint.iter_rows(named=True):
        m_topic = row.get("Topic", "")
        m_name = row.get("Metric", "")
        m_type = row.get("Metric Type", "")
        m_desc = row.get("Description", "")
        d_source = row.get("Data Source", "")
        v_json = row.get("Variables", "")
        
        bp_comp = str(row.get("Comparison Period", ""))
        bp_curr = str(row.get("Current Period", ""))
        
        if not m_name: continue
        
        start_time = time.time()
        
        geo_data, years_ctx, is_cat = calculate_metric_data(master_df, df_pyr, m_name, v_json, m_type)
        if not geo_data:
            continue
            
        focus_geo = geo_data['Focus']['Name']
        broad_geo = geo_data.get('Broad', {}).get('Name', '')
        bench_geo = geo_data.get('Benchmark', {}).get('Name', '')
        peer_geo = geo_data.get('Peer', {}).get('Name', '')
        
        math_debug_logs = []

        # --- GENERATE + GRAMMAR + GEOGRAPHY STANDARDIZATION ---
        p_int, m_int = create_prompt("Internal", geo_data, m_name, m_desc, years_ctx, is_cat, m_type)
        i_int_raw = get_ollama_text(p_int, as_json=True, json_key="insight")
        i_int_grammar = apply_grammar_layer(i_int_raw)
        i_int = apply_geography_standardization_layer(i_int_grammar, [focus_geo])
        
        if broad_geo:
            p_brd, m_brd = create_prompt("Broad", geo_data, m_name, m_desc, years_ctx, is_cat, m_type)
            math_debug_logs.append(f"Broad: {m_brd}")
            i_brd_raw = get_ollama_text(p_brd, as_json=True, json_key="insight")
            i_brd_grammar = apply_grammar_layer(i_brd_raw)
            i_brd = apply_geography_standardization_layer(i_brd_grammar, [focus_geo, broad_geo])
        else: i_brd = "N/A"
        
        if bench_geo:
            p_bnc, m_bnc = create_prompt("Benchmark", geo_data, m_name, m_desc, years_ctx, is_cat, m_type)
            math_debug_logs.append(f"Bench: {m_bnc}")
            i_bnc_raw = get_ollama_text(p_bnc, as_json=True, json_key="insight")
            i_bnc_grammar = apply_grammar_layer(i_bnc_raw)
            i_bnc = apply_geography_standardization_layer(i_bnc_grammar, [focus_geo, bench_geo])
        else: i_bnc = "N/A"
        
        if peer_geo:
            p_per, m_per = create_prompt("Peer", geo_data, m_name, m_desc, years_ctx, is_cat, m_type)
            math_debug_logs.append(f"Peer Avg: {m_per}")
            i_per_raw = get_ollama_text(p_per, as_json=True, json_key="insight")
            i_per_grammar = apply_grammar_layer(i_per_raw)
            i_per = apply_geography_standardization_layer(i_per_grammar, [focus_geo, peer_geo])
            
            # --- Additional detailed peer comparison ---
            p_per_det, m_per_det = create_prompt("Peer_Detailed", geo_data, m_name, m_desc, years_ctx, is_cat, m_type)
            
            print(f"  [Math Debug] {m_per_det}")
            math_debug_logs.append(f"Peer Detailed: {m_per_det}")
            
            i_per_det_raw = get_ollama_text(p_per_det, as_json=True, json_key="insight")
            i_per_det_grammar = apply_grammar_layer(i_per_det_raw)
            
            all_peer_names = [p["Name"] for p in geo_data.get('Peer_Details', [])]
            valid_geos_det = [focus_geo, peer_geo] + all_peer_names
            i_per_det = apply_geography_standardization_layer(i_per_det_grammar, valid_geos_det)
        else: 
            i_per = "N/A"
            i_per_det = "N/A"
        
        cy = years_ctx.get("latest", "")
        py = years_ctx.get("prev", "")
        
        if cy:
            bp_comp = bp_comp.replace("[Current Year]", cy)
            bp_curr = bp_curr.replace("[Current Year]", cy)
        if py:
            bp_comp = bp_comp.replace("[Previous Year]", py)
            bp_curr = bp_curr.replace("[Previous Year]", py)
            
        # --- CHEAT SHEET + SYNTHESIS EXAMPLES ---
        fv_focus = geo_data.get('Focus', {}).get('Formatted_Value', 'N/A')
        fv_broad = geo_data.get('Broad', {}).get('Formatted_Value', 'N/A')
        fv_bench = geo_data.get('Benchmark', {}).get('Formatted_Value', 'N/A')
        fv_peer = geo_data.get('Peer', {}).get('Formatted_Value', 'N/A')

        synth = f"""
        You are an executive data analyst. Synthesize these 4 insights about {m_name} for {cy} into ONE coherent, professional summary sentence.
        
        REFERENCE VALUES (Use these to determine relationships):
        - Focus ({focus_geo}): {fv_focus}
        - Broad ({broad_geo}): {fv_broad}
        - Benchmarks ({bench_geo}): {fv_bench}
        - Peers ({peer_geo}): {fv_peer}

        Insights to synthesize:
        1. Internal: {i_int} 
        2. Broad: {i_brd} 
        3. Benchmarks: {i_bnc} 
        4. Peers: {i_per}
        
        CRITICAL RULES:
        1. DO NOT merge different geographic names together. Group similar relationships elegantly (e.g. "{focus_geo} is higher than X, Y, and Z").
        2. DO NOT compare a geography to itself. NEVER say '{focus_geo}' is higher/lower than '{focus_geo}'.
        3. DO NOT include specific mathematical differences or values in this summary. Keep it qualitative.
        4. Start directly with "{focus_geo}".
        5. Explicitly state the metric being discussed.
        6. Ensure the final sentence is grammatically flawless and DOES NOT have repetitive lists of geographies.
        
        Output STRICTLY as a valid JSON object: {{ "overall_insight": "your sentence here" }}
        """
        
        i_over_raw = get_ollama_text(synth, as_json=True, json_key="overall_insight") if i_int != "N/A" else "N/A"
        i_over_grammar = apply_grammar_layer(i_over_raw)
        
        # Ensure standard Benchmarks are permitted to survive the geography copyeditor
        valid_geos_over = [focus_geo, broad_geo, bench_geo, peer_geo, "Texas", "United States", "US"] + all_peer_names
        i_over = apply_geography_standardization_layer(i_over_grammar, valid_geos_over)
        
        end_time = time.time()
        processing_time = round(end_time - start_time, 2)
        total_processing_time += processing_time
        metrics_processed += 1
        
        print(f"\n[✓] Processed Metric {metrics_processed}: '{m_name}' in {processing_time}s")
        print(f"  [Internal]   {i_int}")
        print(f"  [Broad]      {i_brd}")
        print(f"  [Benchmarks] {i_bnc}")
        print(f"  [Peers]      {i_per}")
        print(f"  [Peers Det]  {i_per_det}")
        print(f"  [Overall]    {i_over}")
        print("-" * 75)
        
        final_results.append({
            "Topic": m_topic,
            "Comparison Period": bp_comp, 
            "Current Period": bp_curr,    
            "Data Source": d_source,
            "Variables": v_json,
            "Metric": m_name,
            "Metric Type": m_type,
            "Description": m_desc,
            "Internal Insight": i_int,
            "Comparative Insight (Broad)": i_brd,
            "Comparative Insight (Benchmarks)": i_bnc,
            "Comparative Insight (Peers)": i_per,
            "Comparative Insight (Peers - Detailed)": i_per_det,
            "Overall Insight": i_over,
            "Math/Prompt Debug": " | ".join(math_debug_logs)
        })

    if metrics_processed == 0: return pd.DataFrame()

    df_final = pd.DataFrame(final_results)

    # --- TOPIC SUMMARIES ---
    print("\nSynthesizing Topic Summaries...")
    topic_summaries = {}
    for topic, group in df_final.groupby("Topic"):
        if not topic: continue
        all_overall_insights = group["Overall Insight"].dropna().tolist()
        insights_str = "\n".join([f"- {insight}" for insight in all_overall_insights if insight != "N/A"])
        
        if not insights_str: 
            topic_summaries[topic] = "N/A"
            continue

        topic_prompt = f"""You are an executive data analyst writing a single cohesive summary paragraph for the topic: {topic}.
Synthesize the following key metrics into a smooth, professional paragraph.
Avoid bullet points. Ensure transitions between sentences feel natural.

Data Points:
{insights_str}

Output STRICTLY as a valid JSON object formatted as: {{ "topic_summary": "your paragraph here" }}"""
        
        raw_summary = get_ollama_text(topic_prompt, as_json=True, json_key="topic_summary")
        polished_summary = apply_grammar_layer(raw_summary)
        topic_summaries[topic] = polished_summary
        
        print(f"  [{topic}] Summary generated.\n")

    # Map back to main dataframe
    df_final["Topic Summary"] = df_final["Topic"].map(topic_summaries).fillna("N/A")

    # --- COMPLETE SUMMARY ---
    print("Synthesizing Complete Executive Summary...")
    all_topics_combined = "\n".join([f"{t}: {s}" for t, s in topic_summaries.items() if s != "N/A"])
    
    complete_summary_prompt = f"""You are an executive data analyst writing a single, high-level executive summary for a dashboard.
Synthesize the following topic-level summaries into ONE cohesive paragraph that highlights the most critical insights across all topics.
Do not use bullet points. Keep it professional, objective, and insightful.

Topic Summaries:
{all_topics_combined}

Output STRICTLY as a valid JSON object formatted as: {{ "complete_summary": "your executive summary here" }}"""

    complete_raw = get_ollama_text(complete_summary_prompt, as_json=True, json_key="complete_summary")
    complete_polished = apply_grammar_layer(complete_raw)
    
    print(f"  [Complete Summary] Summary generated.\n")
    df_final["Complete Summary"] = complete_polished

    columns_ordered = [
        "Topic", "Topic Summary", "Complete Summary", "Comparison Period", "Current Period", 
        "Data Source", "Variables", "Metric", "Metric Type", "Description", 
        "Internal Insight", "Comparative Insight (Broad)", 
        "Comparative Insight (Benchmarks)", "Comparative Insight (Peers)", 
        "Comparative Insight (Peers - Detailed)", "Overall Insight", "Math/Prompt Debug"
    ]
    
    # Reorder columns, ignoring any missing ones gracefully
    df_final = df_final[[col for col in columns_ordered if col in df_final.columns]]
    
    avg_time = round(total_processing_time / metrics_processed, 2)
    print("\n" + "="*50)
    print(f"PIPELINE COMPLETE: {metrics_processed} Metrics processed.")
    print(f"Average Processing Time: {avg_time} seconds/metric")
    print("="*50 + "\n")
    
    df_final.to_csv("dashboard_data_debug_v2.csv", index=False)
    return df_final

if __name__ == "__main__":
    df = main()

Loading data and Blueprint...
Blueprint loaded. Iterating through defined metrics...

  [Math Debug] Focus (Travis County): 1,389,670.0000 | Tol: +/-27,793.4000 | [WFSRCA Region (1,369,412.0000): SIM] [San Antonio MSA (2,813,140.0000): Focus<Peer] [Houston MSA (7,904,627.0000): Focus<Peer] [Dallas MSA (8,477,157.0000): Focus<Peer] 

[✓] Processed Metric 1: 'Population' in 40.78s
  [Internal]   Travis County's population is 1.39 million.
  [Broad]      Travis County's population of 1.39 million represents approximately 53.0% of the Austin MSA's population of 2.62 million.
  [Benchmarks] Travis County's population of 1.39 million represents approximately 4.4% of the population of Texas and US.
  [Peers]      Travis County's population of 1.39 million represents approximately 27.0% of the size of its peer average of 5.14 million.
  [Peers Det]  Travis County’s population of 1.39 million is evaluated against individual peers. Specifically, it is smaller than Dallas MSA (8.48 million), Hous

In [26]:
for i in df['Topic'].unique():
    print("\n\n",i,"\n",df[df['Topic'] == i]['Topic Summary'].iloc[0])



 Population 
 Travis County’s population continues to exhibit robust growth, representing a substantial contributor to the Austin Metropolitan Statistical Area and the state of texas, although it currently remains below the average population levels of comparable regions. While its growth rate significantly outpaces national and texas averages, it trails behind major metropolitan areas like houston and dallas, and lags behind the broader austin msa, wfsrca region, and san antonio msa. Notably, travis county’s cumulative percentage population change since 1990 is substantially higher than the us and texas averages, indicating a particularly accelerated growth trajectory that distinguishes it within the broader demographic landscape.


 Drivers of Change 
 Travis County’s demographic trends reveal a distinct pattern of change, primarily shaped by natural population growth between 1990 and 2025, which substantially outpaced national and state averages, as well as those of its peer regio

In [27]:
df['Complete Summary'].iloc[0]

'Travis County’s demographic profile presents a unique and accelerating growth trajectory, driven primarily by natural population increases and characterized by a notably aging population, particularly among the 75-79 age group, which diverges significantly from national and regional trends and contributes to a higher median age than the Austin MSA and its peers. While the workforce remains highly skilled and engaged, with exceptional educational attainment and labor force participation rates, challenges persist regarding affordability, as evidenced by elevated cost-burden rates for both renter and owner households relative to the Austin MSA and broader Texas, alongside a persistent poverty rate exceeding national and regional benchmarks. Despite robust overall population growth, the county’s expansion lags behind major metropolitan areas, indicating a localized growth pattern that warrants further investigation and strategic planning to address socioeconomic disparities and ensure sus

In [28]:
df['Math/Prompt Debug'].iloc[0]

'Broad: MATH LOCK: Travis County is exactly 1.39 million and Austin MSA is exactly 2.62 million. DO NOT confuse or swap these values. REQUIRED FACT: You must state it is approximately 53.0% the size of Austin MSA. | Bench: MATH LOCK: Travis County is exactly 1.39 million and Texas is exactly 31.71 million. DO NOT confuse or swap these values. REQUIRED FACT: You must state it is approximately 4.4% the size of Texas. | Peer Avg: MATH LOCK: Travis County is exactly 1.39 million and its peer average is exactly 5.14 million. DO NOT confuse or swap these values. REQUIRED FACT: You must state it is approximately 27.0% the size of its peer average. | Peer Detailed: Focus (Travis County): 1,389,670.0000 | Tol: +/-27,793.4000 | [WFSRCA Region (1,369,412.0000): SIM] [San Antonio MSA (2,813,140.0000): Focus<Peer] [Houston MSA (7,904,627.0000): Focus<Peer] [Dallas MSA (8,477,157.0000): Focus<Peer] '